<p align="center">
  <img src="https://i0.wp.com/www.tiempodecine.co/web/wp-content/uploads/2015/11/Robert-De-Niro-in-Taxi-Driver-1976.jpg?resize=750%2C375&ssl=1" style="width:100%; max-width:900px; height:180px; object-fit:cover; border-radius:10px;"/>
</p>
<div style="text-align:center;">
  <h1 style="color:#FFD700; display:inline-block; margin:0;">Optimización del Transporte en Nueva York</h1>
  <p>
    <b>Green Taxi | Machine Learning & Data Science | CRISP-DM</b><br>
    <span style="font-size:1.1em;">Análisis y predicción de tarifas y duración de viajes usando datos reales de taxis verdes de NYC.</span>
  </p>
</div>

# FASE 1: Business Understanding (CRISP-DM)

## Propósito de la Fase de Comprensión del Negocio

En esta primera fase del proceso CRISP-DM, nos enfocamos en entender los objetivos y requisitos del negocio relacionados con el proyecto de análisis de datos. Esto incluye la identificación de los problemas clave que se desean resolver, la definición de los objetivos del proyecto y la comprensión del contexto empresarial en el que se aplicarán los resultados del análisis. El objetivo es asegurar que el proyecto esté alineado con las necesidades del negocio y que los resultados sean relevantes y útiles para la toma de decisiones.

# Carga de Librerías

In [17]:
# Verifica si las librerías necesarias ya están instaladas
try:
    # Ignorar warnings
    import warnings
    warnings.filterwarnings('ignore')
    # Librerías del sistema
    import sys, subprocess
    # Importaciones base
    import pandas as pd
    import numpy as np
    import matplotlib.pyplot as plt
    import seaborn as sns
    import scipy.stats as stats
    # Librerías de sistema de archivos
    import os as os  
    from os import path
    import pickle as pkl
    import joblib
    import duckdb

except ImportError:

    print("Dependencias no encontradas. Instalando ahora...")
    
    # Se ejecutan los comandos de instalación
    %pip install --quiet matplotlib
    %pip install --quiet seaborn
    %pip install --quiet joblib
    %pip install --quiet scipy
    %pip install --quiet duckdb
    %pip install --quiet pyarrow
    %pip install --quiet fastparquet
    
    print("Instalación completada.")

# Carga del Dataset Completo Trip Data  FHVHV 06-2025

In [18]:
# --- 1. CONFIGURACIÓN DEL PIPELINE (CORREGIDA) ---

# Llaves corregidas para que coincidan con los datos (Jul, Ago, Sep)
FHVHV_URLS = {
    "fhvhv_julio_2025": "https://d37ci6vzurychx.cloudfront.net/trip-data/fhvhv_tripdata_2025-07.parquet",
    "fhvhv_agosto_2025": "https://d37ci6vzurychx.cloudfront.net/trip-data/fhvhv_tripdata_2025-08.parquet",
    "fhvhv_septiembre_2025": "https://d37ci6vzurychx.cloudfront.net/trip-data/fhvhv_tripdata_2025-09.parquet"
}

# Llaves corregidas para ser únicas y descriptivas (ambas son de Julio)
TAXIS_URLS = {
    "yellow_julio_2025": "https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2025-07.parquet",
    "green_julio_2025": "https://d37ci6vzurychx.cloudfront.net/trip-data/green_tripdata_2025-07.parquet",
}

# El interruptor sigue igual
MODO_DESARROLLO = True  # Cambiar a False para producción

In [19]:
# --- 2. LA FUNCIÓN DEL PIPELINE ---
def crear_dataset_final(fuentes_url: dict, modo_dev: bool = True):
    """
    Carga y concatena múltiples archivos Parquet desde URLs.
    
    - Si modo_dev=True: Toma una muestra rápida del 1% de cada archivo.
    - Si modo_dev=False: Carga el 100% de cada archivo.
    """
    lista_dataframes = []
    
    if modo_dev:
        print(f"--- MODO DESARROLLO (Muestra: 1%) ---")
        sql_sample_clause = "USING SAMPLE 1%"
    else:
        print(f"--- MODO PRODUCCIÓN (Datos: 100%) ---")
        sql_sample_clause = "" # Sin muestreo

    con = duckdb.connect(database=':memory:')
    
    for nombre_fuente, url in fuentes_url.items():
        print(f"Procesando: {nombre_fuente}...")
        
        query = f"""
        SELECT *
        FROM '{url}'
        {sql_sample_clause}
        """
        
        try:
            df_temp = con.execute(query).fetch_df()
            lista_dataframes.append(df_temp)
            print(f"   -> Filas cargadas: {len(df_temp)}")
        except Exception as e:
            print(f"   -> ERROR al procesar {url}: {e}")

    con.close()
    
    if not lista_dataframes:
        print("No se cargaron datos.")
        return pd.DataFrame()

    print("\nConcatenando DataFrames...")
    df_final = pd.concat(lista_dataframes, axis=0, ignore_index=True)
    
    print(f"¡Proceso completado! Total de filas: {len(df_final)}")
    return df_final

# Creación de Muestra 1% del Dataset FHVHV - Taxi Verde & Taxi Amarillo

In [20]:
# --- 3. EJECUCIÓN DEL PIPELINE ---

print("=== Iniciando Pipeline para FHVHV (Uber/Lyft) ===")
df_fhvhv_sample = crear_dataset_final(FHVHV_URLS, modo_dev=MODO_DESARROLLO)

print("\n" + "="*40 + "\n")

print("=== Iniciando Pipeline para TAXIS (Yellow/Green) ===")
df_taxis_sample = crear_dataset_final(TAXIS_URLS, modo_dev=MODO_DESARROLLO)

print("\n" + "="*40 + "\n")
print("Pipelines finalizados. Tienes dos DataFrames de muestra:")
print(f"1. df_fhvhv_sample ({len(df_fhvhv_sample)} filas)")
print(f"2. df_taxis_sample ({len(df_taxis_sample)} filas)")

=== Iniciando Pipeline para FHVHV (Uber/Lyft) ===
--- MODO DESARROLLO (Muestra: 1%) ---
Procesando: fhvhv_julio_2025...
   -> Filas cargadas: 178176
Procesando: fhvhv_agosto_2025...
   -> Filas cargadas: 178176
Procesando: fhvhv_septiembre_2025...
   -> Filas cargadas: 202752

Concatenando DataFrames...
¡Proceso completado! Total de filas: 559104


=== Iniciando Pipeline para TAXIS (Yellow/Green) ===
--- MODO DESARROLLO (Muestra: 1%) ---
Procesando: yellow_julio_2025...
   -> Filas cargadas: 32768
Procesando: green_julio_2025...
   -> Filas cargadas: 0

Concatenando DataFrames...
¡Proceso completado! Total de filas: 32768


Pipelines finalizados. Tienes dos DataFrames de muestra:
1. df_fhvhv_sample (559104 filas)
2. df_taxis_sample (32768 filas)


# Carga del Dataset FHVHV (Uber/Lyft)  & TAXIS (Yellow/Green)

### FHVHV (Uber/Lyft)

In [21]:
# FHVHV (Uber/Lyft)
print("Guardando muestra de FHVHV (Uber/Lyft)...")
archivo_fhvhv = 'fhvhv_trimestral.parquet'

try:
    df_fhvhv_sample.to_parquet(archivo_fhvhv, index=False)
    print(f"-> ¡Éxito! Guardado como '{archivo_fhvhv}'")
except Exception as e:
    print(f"-> ERROR al guardar FHVHV: {e}")

Guardando muestra de FHVHV (Uber/Lyft)...
-> ¡Éxito! Guardado como 'fhvhv_trimestral.parquet'


### TAXIS (Yellow/Green)


In [22]:
# TAXIS (Yellow/Green)
print("\nGuardando muestra de TAXIS (Yellow/Green)...")
archivo_taxis = 'taxis_julio.parquet'

try:
    # (Nota: este aún no está normalizado, ¡es el siguiente paso!)
    df_taxis_sample.to_parquet(archivo_taxis, index=False)
    print(f"-> ¡Éxito! Guardado como '{archivo_taxis}'")
except Exception as e:
    print(f"-> ERROR al guardar Taxis: {e}")


Guardando muestra de TAXIS (Yellow/Green)...
-> ¡Éxito! Guardado como 'taxis_julio.parquet'


In [23]:
taxis_sample = pd.read_parquet('taxis_julio.parquet')
fhvhv_sample = pd.read_parquet('fhvhv_trimestral.parquet')